# Tuned supervised-only baselines at 70 labels (Table 12, Section IV-M; reviewer R2-C4)

The published labels-only baseline fine-tunes MobileNetV3-Small on the 70 labelled leaves with the student's own recipe
(Adam, lr 1e-3, 30 epochs, batch 64), which is only ~60 gradient steps. This notebook adds **pre-declared, stronger
baselines** on the **same ten label draws** and compares each, paired by draw, with probe + distillation:

1. **Fine-tune, lr 1e-4** — AdamW, 150 epochs, batch 16, strong augmentation (random resized crop, flips, rotation, colour jitter).
2. **Fine-tune, lr 3e-4** — same, higher learning rate.
3. **Linear probe on frozen ImageNet MobileNetV3-Small features** — the same 70-label multinomial probe (C = 1) as the DINO teacher.
4. **Linear probe on frozen ImageNet ResNet-50 features** — a stronger ImageNet backbone.

All configurations were fixed before running and **all are reported**; none is chosen on the test set.
Probe + distillation scores are read from the released Table 12 files, not retrained.
Resumable: re-run after a disconnect and finished draws are skipped. Expected GPU time ~30–45 min (T4/L4).

Output: `baseline_tuned.json` + `baseline_tuned_scores.npz` — upload both.

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'                              # one folder per class
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
ESCORES    = f'{RES}/E_scores_5rep.npz'          # published Experiment E, draws 0-4
ESCORES2   = f'{RES}/E_scores_rep5_9.npz'        # Experiment E, draws 5-9 (more_seeds.ipynb)
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
REPS       = list(range(10))                     # the same ten label draws as Table 12
OUT_NPZ    = f'{RES}/baseline_tuned_scores.npz'
OUT_JSON   = f'{RES}/baseline_tuned.json'
# ---- pre-declared baselines (fixed before any result is seen; ALL are reported, none is selected on the test set) ----
FT_CONFIGS = {                                    # fine-tune MobileNetV3-Small (ImageNet init) on the 70 labelled leaves only
    'ft_lr1e-4_strongaug': dict(lr=1e-4, epochs=150, bs=16, aug='strong'),
    'ft_lr3e-4_strongaug': dict(lr=3e-4, epochs=150, bs=16, aug='strong'),
}
PROBE_BACKBONES = ['mobilenet_v3_small', 'resnet50']   # frozen ImageNet features + the same 70-label logistic probe as the DINO probe


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
!pip -q install onnx onnxruntime timm
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


In [ ]:
# CELL 5 — crop cache (JPEG q95, same as the revision_v1 run), parallel
def cache_path(rec):
    return os.path.join(CACHE, hashlib.md5(rec['rel'].encode()).hexdigest() + '.jpg')
def _build(rec):
    cp = cache_path(rec)
    if not os.path.exists(cp):
        try: leaf_crop(rec['path']).save(cp, quality=95)
        except Exception as e: return rec['rel'], str(e)
    return None
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    fails = [f for f in ex.map(_build, RECS, chunksize=32) if f]
print('crop cache: %d images, %d failures, %.0fs' % (len(RECS), len(fails), time.time() - t0))
assert not fails, fails[:5]


In [ ]:
# CELL 6 — tuned baselines on the 70 labelled leaves (binary labels, as the published labels-only baseline)
import torch.nn.functional as Fnn
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
AUG = {'strong': T.Compose([T.RandomResizedCrop(IMG, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                            T.RandomRotation(30), T.ColorJitter(.3, .3, .3, .05)])}
class DS(Dataset):
    def __init__(self, recs, labels, aug=None): self.r, self.l, self.aug = recs, labels, aug
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        im = Image.open(cache_path(self.r[i])).convert('RGB')
        if self.aug is not None: im = self.aug(im)
        return TO_T(im), int(self.l[i])
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def finetune(recs, labels, seed, lr, epochs, bs, aug):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2); m = m.to(device)
    dl = DataLoader(DS(recs, labels, AUG[aug]), batch_size=bs, shuffle=True, num_workers=4, drop_last=False)
    opt = optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(dl)); lf = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for x, y in dl:
            opt.zero_grad(); lf(m(x.to(device)), y.to(device)).backward(); opt.step(); sch.step()
    return m.eval()
@torch.no_grad()
def scores(m, recs):
    dl = DataLoader(DS(recs, np.zeros(len(recs))), batch_size=256, num_workers=4)
    return np.concatenate([torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy() for x, _ in dl])
# the label draws must be the published ones: refit the DINO probe from the cached features and compare with Table 12
_pa = [roc_auc_score(dte, 1 - LogisticRegression(C=1.0, max_iter=2000).fit(F['Xtr'][draw(r)], ytr[draw(r)]).predict_proba(F['Xte'])[:, HEALTHY]) for r in range(5)]
assert TRAIN_ORDER_OK and np.allclose(_pa, [0.9791, 0.9858, 0.9888, 0.9836, 0.9893], atol=1e-3), ('label draws differ from the published run', _pa)
print('label draws reproduce the published probes:', np.round(_pa, 4))
S = {k: v for k, v in np.load(OUT_NPZ).items()} if os.path.exists(OUT_NPZ) else {}
for name, c in FT_CONFIGS.items():
    for rep in REPS:
        key = f'{name}_rep{rep}'
        if key in S: continue
        idx = draw(rep); t0 = time.time()
        m = finetune([TRAIN[i] for i in idx], dtr[idx], seed=3000 + rep, **c)
        S[key] = scores(m, TEST); np.savez(OUT_NPZ, **S); del m; torch.cuda.empty_cache()
        print('%-22s draw %d  AUC %.4f  (%.1f min)' % (name, rep, roc_auc_score(dte, S[key]), (time.time() - t0) / 60))


In [ ]:
# CELL 7 — frozen ImageNet features + the same 70-label multinomial probe (C = 1) as the DINO teacher
@torch.no_grad()
def imagenet_feats(backbone, recs, bs=256):
    if backbone == 'mobilenet_v3_small':
        net = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1); body = nn.Sequential(net.features, net.avgpool)
    else:
        net = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2); body = nn.Sequential(*list(net.children())[:-1])
    body = body.to(device).eval(); out = []
    for i in range(0, len(recs), bs):
        x = torch.stack([TO_T(Image.open(cache_path(r)).convert('RGB')) for r in recs[i:i + bs]]).to(device)
        out.append(Fnn.normalize(body(x).flatten(1).float(), dim=1).cpu().numpy())
    del body, net; torch.cuda.empty_cache(); return np.concatenate(out)
for bb in PROBE_BACKBONES:
    if all(f'probe_{bb}_rep{r}' in S for r in REPS): continue
    t0 = time.time(); Ftr, Fte = imagenet_feats(bb, TRAIN), imagenet_feats(bb, TEST)
    print('%s features %s in %.1f min' % (bb, Ftr.shape, (time.time() - t0) / 60))
    for rep in REPS:
        idx = draw(rep); p = LogisticRegression(C=1.0, max_iter=2000).fit(Ftr[idx], ytr[idx])
        S[f'probe_{bb}_rep{rep}'] = 1.0 - p.predict_proba(Fte)[:, HEALTHY]
        S[f'probe_{bb}_acc7_rep{rep}'] = np.array([(p.predict(Fte) == yte).mean()])
    np.savez(OUT_NPZ, **S)
    print('%s probe AUC: %s' % (bb, np.round([roc_auc_score(dte, S[f'probe_{bb}_rep{r}']) for r in REPS], 4)))


In [ ]:
# CELL 8 — paired comparison with probe + distillation (Table 12), all baselines reported
from scipy.stats import ttest_rel, wilcoxon
E5, E10 = np.load(ESCORES), np.load(ESCORES2)
distil = np.array([roc_auc_score(dte, E5['probe_distil'][r]) for r in range(5)] + [roc_auc_score(dte, E10[f'probe_distil_rep{r}']) for r in range(5, 10)])
lab_only = np.array([roc_auc_score(dte, E5['labels_only'][r]) for r in range(5)] + [roc_auc_score(dte, E10[f'labels_only_rep{r}']) for r in range(5, 10)])
assert abs(distil.mean() - 0.978) < 1e-3 and abs(lab_only.mean() - 0.956) < 1e-3, 'published Table 12 values not reproduced: stop'
rows = {'labels_only_published': lab_only}
for name in FT_CONFIGS: rows[name] = np.array([roc_auc_score(dte, S[f'{name}_rep{r}']) for r in REPS])
for bb in PROBE_BACKBONES: rows[f'probe_imagenet_{bb}'] = np.array([roc_auc_score(dte, S[f'probe_{bb}_rep{r}']) for r in REPS])
OUT = dict(probe_distil=dict(per_draw=distil.tolist(), mean=float(distil.mean()), sd=float(distil.std(ddof=1))), baselines={},
           configs=dict(finetune=FT_CONFIGS, probe_backbones=PROBE_BACKBONES, seed_rule='torch seed 3000+draw; same label draws as Table 12'))
print('%-32s %8s %7s %8s %6s %9s %9s' % ('baseline', 'AUC', 'SD', 'gain', 'wins', 'paired-t', 'Wilcoxon'))
for k, v in rows.items():
    d = distil - v
    OUT['baselines'][k] = dict(per_draw=v.tolist(), mean=float(v.mean()), sd=float(v.std(ddof=1)), min=float(v.min()), max=float(v.max()),
                              gain_mean=float(d.mean()), wins=int((d > 0).sum()), paired_t_p=float(ttest_rel(distil, v).pvalue),
                              wilcoxon_p=float(wilcoxon(d).pvalue))
    if k.startswith('probe_imagenet_'):
        bb = k.replace('probe_imagenet_', ''); OUT['baselines'][k]['acc7'] = float(np.mean([S[f'probe_{bb}_acc7_rep{r}'][0] for r in REPS]))
    b = OUT['baselines'][k]
    print('%-32s %8.4f %7.4f %+8.4f %4d/10 %9.4f %9.4f' % (k, b['mean'], b['sd'], b['gain_mean'], b['wins'], b['paired_t_p'], b['wilcoxon_p']))
best = max((k for k in rows if k != 'labels_only_published'), key=lambda k: rows[k].mean())
OUT['strongest_baseline'] = best
print('\nstrongest baseline (by mean, reported for transparency; not used to pick anything):', best)
json.dump(OUT, open(OUT_JSON, 'w'), indent=1); print('saved', OUT_JSON, 'and', OUT_NPZ, '-> upload both')
